### Setup

In [ ]:
!pip install torchinfo

In [ ]:
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00


In [ ]:
!pip install timm

In [ ]:
from google.colab import drive

import os
import shutil
import time
import random
import cv2
import numpy as np
import scipy.ndimage as ndi
import matplotlib.pyplot as plt

from skimage.exposure import match_histograms
from skimage.morphology import skeletonize, reconstruction, remove_small_objects
from skimage.filters import apply_hysteresis_threshold

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import segmentation_models_pytorch as smp

from torchinfo import summary
from tqdm.auto import tqdm

%matplotlib inline

In [ ]:
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

  # Add these lines to force strict GPU determinism:
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
set_seed(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive_base_dir = '/content/drive/MyDrive/project/tnt'
local_base_dir = '/content/project/tnt'

In [ ]:
local_data_dir = local_base_dir+'/data'
local_model_dir = local_base_dir+'/model'
local_plot_dir = local_base_dir+'/plot'
local_script_dir = local_base_dir+'/script'
os.makedirs(local_data_dir, exist_ok=True)
os.makedirs(local_model_dir, exist_ok=True)
os.makedirs(local_plot_dir, exist_ok=True)
os.makedirs(local_script_dir, exist_ok=True)

In [ ]:
drive_data_dir = drive_base_dir+'/data'
drive_model_dir = drive_base_dir+'/model'
drive_plot_dir = drive_base_dir+'/plot'
drive_script_dir = drive_base_dir+'/script'
os.makedirs(drive_data_dir, exist_ok=True)
os.makedirs(drive_model_dir, exist_ok=True)
os.makedirs(drive_plot_dir, exist_ok=True)
os.makedirs(drive_script_dir, exist_ok=True)

In [ ]:
!jupyter nbconvert --to script '{drive_base_dir}/tnt.ipynb'
!mv '{drive_base_dir}/tnt.txt' '{drive_base_dir}/tnt.py'
!mv '{drive_base_dir}/tnt.py' '{drive_script_dir}'

[NbConvertApp] Converting notebook /content/drive/MyDrive/project/tnt/tnt.ipynb to script
[NbConvertApp] Writing 79159 bytes to /content/drive/MyDrive/project/tnt/tnt.txt


In [ ]:
!jupyter nbconvert --to script '{drive_base_dir}/tnt.ipynb'
!mv '{drive_base_dir}/tnt.txt' '{drive_base_dir}/tnt.py'
!mv '{drive_base_dir}/tnt.py' '{local_script_dir}'

[NbConvertApp] Converting notebook /content/drive/MyDrive/project/tnt/tnt.ipynb to script
[NbConvertApp] Writing 79194 bytes to /content/drive/MyDrive/project/tnt/tnt.txt


In [ ]:
target_dir = local_data_dir

for item in os.listdir(target_dir):
  item_path = os.path.join(target_dir, item)
  if os.path.isdir(item_path):
    shutil.rmtree(item_path)
  else:
    os.remove(item_path)

print(f'Cleared all contents inside: {target_dir}')

Cleared all contents inside: /content/project/tnt/data


In [ ]:
!ls -la {local_data_dir}

total 8
drwxr-xr-x 2 root root 4096 Sep 17 13:22 .
drwxr-xr-x 6 root root 4096 Sep 17 13:22 ..


In [ ]:
!cp -r /content/project/tnt/model/* /content/drive/MyDrive/project/tnt/model/
!cp -r /content/project/tnt/plot/* /content/drive/MyDrive/project/tnt/plot/

cp: cannot stat '/content/project/tnt/model/*': No such file or directory
cp: cannot stat '/content/project/tnt/plot/*': No such file or directory


### Data Extraction

In [ ]:
dataset_splits = {
  'train': (os.path.join(drive_base_dir, 'm05.png'), os.path.join(drive_base_dir, 'm05-label.png')),
  'val':   (os.path.join(drive_base_dir, 'm02.png'), os.path.join(drive_base_dir, 'm02-label.png')),
  'test':  (os.path.join(drive_base_dir, 'm03.png'), os.path.join(drive_base_dir, 'm03-label.png'))
}

In [ ]:
for split_name, (img_path, annot_path) in dataset_splits.items():
  if not os.path.exists(img_path):
    raise FileNotFoundError(f'Missing {split_name} image at: {img_path}')
  if not os.path.exists(annot_path):
    raise FileNotFoundError(f'Missing {split_name} annotation at: {annot_path}')

print('All split paths verified successfully.')

All split paths verified successfully.


In [ ]:
def calibrate_marker_hsv(annot_path):
  annot_bgr = cv2.imread(str(annot_path))
  if annot_bgr is None:
    raise FileNotFoundError(f'Cannot read annotation image at: {annot_path}')

  hsv = cv2.cvtColor(annot_bgr, cv2.COLOR_BGR2HSV)
  colored_pixels = hsv[hsv[:, :, 1] > 35]

  h_low, h_high = np.percentile(colored_pixels[:, 0], [1, 99])
  s_low = np.percentile(colored_pixels[:, 1], 1)
  v_low = np.percentile(colored_pixels[:, 2], 1)

  lower_hsv = np.array([int(max(0, h_low - 5)), int(max(25, s_low - 10)), int(max(25, v_low - 10))])
  upper_hsv = np.array([int(min(179, h_high + 5)), 255, 255])

  return lower_hsv, upper_hsv

In [ ]:
def binarize_annotation(annot_bgr, lower_hsv, upper_hsv, min_area_px=15):
  hsv = cv2.cvtColor(annot_bgr, cv2.COLOR_BGR2HSV)
  raw_mask = cv2.inRange(hsv, lower_hsv, upper_hsv)
  num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(raw_mask, connectivity=8)

  clean_mask = np.zeros_like(raw_mask)
  for i in range(1, num_labels):
    if stats[i, cv2.CC_STAT_AREA] >= min_area_px:
      clean_mask[labels == i] = 255

  return clean_mask

In [ ]:
def patch_extraction(
  dataset_splits,
  output_dir,
  patch_size=512,
  patch_stride=512,
  pos_stride=64,
  neg_stride=128,
  pure_bg_keep_prob=0.05,
  hard_bg_keep_prob=0.20,
  random_seed=42,
  use_calibrated_hsv=True,
  extract_background=True
  ):

  np.random.seed(random_seed)
  os.makedirs(output_dir, exist_ok=True)

  if use_calibrated_hsv:
    lower_hsv, upper_hsv = calibrate_marker_hsv(dataset_splits['train'][1])
  else:
    lower_hsv = (20, 100, 100)
    upper_hsv = (40, 255, 255)

  for split_name, (orig_path, annot_path) in dataset_splits.items():
    raw_gray = cv2.imread(str(orig_path), cv2.IMREAD_GRAYSCALE)
    annot_bgr = cv2.imread(str(annot_path), cv2.IMREAD_COLOR)
    clean_mask = binarize_annotation(annot_bgr, lower_hsv, upper_hsv, min_area_px=15)

    img_dir = os.path.join(output_dir, split_name, 'images')
    mask_dir = os.path.join(output_dir, split_name, 'masks')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    h, w = raw_gray.shape

    if split_name == 'train' and extract_background:
      for y in range(0, h - patch_size + 1, pos_stride):
        for x in range(0, w - patch_size + 1, pos_stride):
          img_patch = raw_gray[y:y+patch_size, x:x+patch_size]
          mask_patch = clean_mask[y:y+patch_size, x:x+patch_size]
          if np.sum(mask_patch > 0) > 15:
            cv2.imwrite(os.path.join(img_dir, f'train_pos_y{y}_x{x}.png'), img_patch)
            cv2.imwrite(os.path.join(mask_dir, f'train_pos_y{y}_x{x}.png'), mask_patch)

      sobel_x = cv2.Sobel(raw_gray, cv2.CV_32F, 1, 0, ksize=3)
      sobel_y = cv2.Sobel(raw_gray, cv2.CV_32F, 0, 1, ksize=3)
      edge_map = cv2.magnitude(sobel_x, sobel_y)
      edge_thresh = float(np.percentile(edge_map, 75))

      for y in range(0, h - patch_size + 1, neg_stride):
        for x in range(0, w - patch_size + 1, neg_stride):
          img_patch = raw_gray[y:y+patch_size, x:x+patch_size]
          mask_patch = clean_mask[y:y+patch_size, x:x+patch_size]
          if np.sum(mask_patch > 0) == 0:
            is_hard = np.mean(edge_map[y:y+patch_size, x:x+patch_size]) > edge_thresh
            roll = np.random.rand()
            if (is_hard and roll < hard_bg_keep_prob) or (not is_hard and roll < pure_bg_keep_prob):
              tag = 'hardbg' if is_hard else 'purebg'
              cv2.imwrite(os.path.join(img_dir, f'train_{tag}_y{y}_x{x}.png'), img_patch)
              cv2.imwrite(os.path.join(mask_dir, f'train_{tag}_y{y}_x{x}.png'), mask_patch)

    else:
      for y in range(0, h - patch_size + 1, patch_stride):
        for x in range(0, w - patch_size + 1, patch_stride):
          img_patch = raw_gray[y:y+patch_size, x:x+patch_size]
          mask_patch = clean_mask[y:y+patch_size, x:x+patch_size]
          cv2.imwrite(os.path.join(img_dir, f'{split_name}_grid_y{y}_x{x}.png'), img_patch)
          cv2.imwrite(os.path.join(mask_dir, f'{split_name}_grid_y{y}_x{x}.png'), mask_patch)

In [ ]:
def audit_patches(output_dir):

  if not os.path.exists(output_dir):
    print(f'Directory {output_dir} does not exists')
    return

  splits = os.listdir(output_dir)
  for split in sorted(splits):
    print(f'[Folder {split}]:')
    split_path = os.path.join(output_dir, split)
    if not os.path.isdir(split_path):
      continue

    img_dir = os.path.join(split_path, 'images')
    mask_dir = os.path.join(split_path, 'masks')

    if not os.path.exists(img_dir):
      print('No images folder found')
      continue

    filenames = os.listdir(img_dir)
    total_patches = len(filenames)

    pos_count = sum(1 for f in filenames if '_pos_' in f)
    hardbg_count = sum(1 for f in filenames if '_hardbg_' in f)
    purebg_count = sum(1 for f in filenames if '_purebg_' in f)
    grid_count = sum(1 for f in filenames if '_grid_' in f)

    print(f'Total pacthes saved: {total_patches}')
    if pos_count > 0: print(f'Positive Nanotube Patches: {pos_count}')
    if hardbg_count > 0: print(f'Hard-Negative Backgrounds: {hardbg_count}')
    if purebg_count > 0: print(f'Pure Backgrounds: {purebg_count}')
    if grid_count > 0: print(f'Grid Evaluation Patches: {grid_count}')

    img_files = set(filenames)
    mask_files = set(os.listdir(mask_dir) if os.path.exists(mask_dir) else set())
    missing_masks = img_files - mask_files

    if missing_masks:
      print(f'Found {len(missing_masks)} images missing matching masks.')
    else:
      print(f'All images have exact matching masks.')

In [ ]:
patch_extraction(
  dataset_splits,
  local_data_dir,
  patch_size=512,
  patch_stride=512,
  pos_stride=64,
  neg_stride=128,
  pure_bg_keep_prob=0.05,
  hard_bg_keep_prob=0.20,
  random_seed=42,
  use_calibrated_hsv=True,
  extract_background=True
)

In [ ]:
audit_patches(local_data_dir)

[Folder test]:
Total pacthes saved: 108
Grid Evaluation Patches: 108
All images have exact matching masks.
[Folder train]:
Total pacthes saved: 2209
Positive Nanotube Patches: 2043
Hard-Negative Backgrounds: 146
Pure Backgrounds: 20
All images have exact matching masks.
[Folder val]:
Total pacthes saved: 108
Grid Evaluation Patches: 108
All images have exact matching masks.


### Dataset, Transform, Dataloader

In [ ]:
class MicroscopicDataset(Dataset):
  def __init__(self, split_dir, transform=None):
    self.image_dir = os.path.join(split_dir, 'images')
    self.mask_dir = os.path.join(split_dir, 'masks')

    if not os.path.exists(self.image_dir):
      raise FileNotFoundError(f'Image directory not found: {self.image_dir}')
    if not os.path.exists(self.mask_dir):
      raise FileNotFoundError(f'Mask directory not found: {self.mask_dir}')

    self.filenames = sorted([f for f in os.listdir(self.image_dir) if f.endswith(('.png', '.jpg', 'jpeg'))])
    self.transform = transform

  def __len__(self):
    return len(self.filenames)

  def __getitem__(self, idx):
    fname = self.filenames[idx]
    image = cv2.imread(os.path.join(self.image_dir, fname), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(os.path.join(self.mask_dir, fname), cv2.IMREAD_GRAYSCALE)

    if image is None or mask is None:
      raise FileNotFoundError(f'Failed to read image or mask for:  {fname}')

    image = np.expand_dims(image, axis=-1)
    mask = np.expand_dims(mask, axis=-1)

    if self.transform is not None:
      augmented = self.transform(image=image, mask=mask)
      image_tensor = augmented['image']
      mask_tensor = augmented['mask']
    else:
      image_tensor = torch.from_numpy(image).float().permute(2, 0, 1) / 255.0
      mask_tensor = torch.from_numpy(mask).float().permute(2, 0, 1)

    image_tensor = image_tensor.repeat(3, 1, 1)

    if mask_tensor.ndim == 3 and mask_tensor.shape[-1] == 1:
      mask_tensor = mask_tensor.permute(2, 0, 1)
    elif mask_tensor.ndim == 2:
      mask_tensor = mask_tensor.unsqueeze(0)

    mask_tensor = (mask_tensor > 127.0).float()

    return image_tensor, mask_tensor

In [ ]:
train_transform = A.Compose([
  A.HorizontalFlip(p=0.5),
  A.RandomRotate90(p=0.5),
  A.RandomGamma(gamma_limit=(85, 115), p=0.5),
  A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
  A.Normalize(mean=(0.0,), std=(1.0,)),
  ToTensorV2()
])

In [ ]:
val_test_transform = A.Compose([
  A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
  A.Normalize(mean=(0.0,), std=(1.0,)),
  ToTensorV2()
])

In [ ]:
BATCH_SIZE = 8

In [ ]:
train_dataset = MicroscopicDataset(os.path.join(local_data_dir, 'train'), transform=train_transform)
val_dataset = MicroscopicDataset(os.path.join(local_data_dir, 'val'), transform=val_test_transform)
test_dataset = MicroscopicDataset(os.path.join(local_data_dir, 'test'), transform=val_test_transform)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size = BATCH_SIZE,shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size = BATCH_SIZE,shuffle=False, num_workers=0, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size = BATCH_SIZE,shuffle=False, num_workers=0, pin_memory=True)

### Model

In [ ]:
model_p1 = smp.Unet(
  encoder_name='resnet34',
  encoder_weights='imagenet',
  in_channels=3,
  classes=1
).to(device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
summary(model_p1)

Layer (type:depth-idx)                        Param #
Unet                                          --
├─ResNetEncoder: 1-1                          --
│    └─Conv2d: 2-1                            9,408
│    └─BatchNorm2d: 2-2                       128
│    └─ReLU: 2-3                              --
│    └─MaxPool2d: 2-4                         --
│    └─Sequential: 2-5                        --
│    │    └─BasicBlock: 3-1                   73,984
│    │    └─BasicBlock: 3-2                   73,984
│    │    └─BasicBlock: 3-3                   73,984
│    └─Sequential: 2-6                        --
│    │    └─BasicBlock: 3-4                   230,144
│    │    └─BasicBlock: 3-5                   295,424
│    │    └─BasicBlock: 3-6                   295,424
│    │    └─BasicBlock: 3-7                   295,424
│    └─Sequential: 2-7                        --
│    │    └─BasicBlock: 3-8                   919,040
│    │    └─BasicBlock: 3-9                   1,180,672
│    │    └─Basi

In [ ]:
model_p2 = smp.Unet(
    encoder_name='tu-convnext_nano',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    decoder_use_batchnorm=True
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 62.4MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
summary(model_p2)

Layer (type:depth-idx)                             Param #
Unet                                               --
├─TimmUniversalEncoder: 1-1                        --
│    └─FeatureListNet: 2-1                         --
│    │    └─Conv2d: 3-1                            3,920
│    │    └─LayerNorm2d: 3-2                       160
│    │    └─ConvNeXtStage: 3-3                     111,680
│    │    └─ConvNeXtStage: 3-4                     479,680
│    │    └─ConvNeXtStage: 3-5                     6,907,520
│    │    └─ConvNeXtStage: 3-6                     7,448,320
├─UnetDecoder: 1-2                                 --
│    └─Identity: 2-2                               --
│    └─ModuleList: 2-3                             --
│    │    └─UnetDecoderBlock: 3-7                  2,802,688
│    │    └─UnetDecoderBlock: 3-8                  627,200
│    │    └─UnetDecoderBlock: 3-9                  156,928
│    │    └─UnetDecoderBlock: 3-10                 27,776
│    │    └─UnetDecoderBlock

In [ ]:
model_p3 = smp.Unet(
    encoder_name='mit_b3',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    decoder_use_batchnorm=True
).to(device)

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  178MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
summary(model_p3)

Layer (type:depth-idx)                        Param #
Unet                                          --
├─MixVisionTransformerEncoder: 1-1            --
│    └─OverlapPatchEmbed: 2-1                 --
│    │    └─Conv2d: 3-1                       9,472
│    │    └─LayerNorm: 3-2                    128
│    └─OverlapPatchEmbed: 2-2                 --
│    │    └─Conv2d: 3-3                       73,856
│    │    └─LayerNorm: 3-4                    256
│    └─OverlapPatchEmbed: 2-3                 --
│    │    └─Conv2d: 3-5                       368,960
│    │    └─LayerNorm: 3-6                    640
│    └─OverlapPatchEmbed: 2-4                 --
│    │    └─Conv2d: 3-7                       1,475,072
│    │    └─LayerNorm: 3-8                    1,024
│    └─Sequential: 2-5                        --
│    │    └─Block: 3-9                        314,880
│    │    └─Block: 3-10                       314,880
│    │    └─Block: 3-11                       314,880
│    └─LayerNorm: 2-6   

### Loss Function, Optimizer

In [ ]:
class SoftSkeletonize(nn.Module):
  def __init__(self, iterations=3):
    super().__init__()
    self.iterations = iterations

  def soft_erode(self, img):
    p1 = -F.max_pool2d(-img, (3, 1), stride=1, padding=(1, 0))
    p2 = -F.max_pool2d(-img, (1, 3), stride=1, padding=(0, 1))
    return torch.min(p1, p2)

  def soft_dilate(self, img):
    return F.max_pool2d(img, (3, 3), stride=1, padding=1)

  def soft_open(self, img):
    return self.soft_dilate(self.soft_erode(img))

  def forward(self, img):
    img = torch.clamp(img, 0.0, 1.0)
    skel = F.relu(img - self.soft_open(img))
    for _ in range(self.iterations):
      img = self.soft_erode(img)
      skel = skel + F.relu(img - self.soft_open(img))
    return torch.clamp(skel, 0.0, 1.0)

In [ ]:
class HybridSoftClDiceLoss(nn.Module):
  def __init__(self, bce_weight=1.0, cldice_weight=3.0, pos_weight=4.0):
    super().__init__()
    self.bce_weight = bce_weight
    self.cldice_weight = cldice_weight
    self.soft_skel = SoftSkeletonize(iterations=3)
    self.pos_weight = pos_weight

  def forward(self, logits, targets):
    probs = torch.sigmoid(logits)
    weights = 1.0 + (self.pos_weight - 1.0) * targets
    bce = F.binary_cross_entropy(probs, targets, weight=weights)
    skel_pred = self.soft_skel(probs)
    skel_true = self.soft_skel(targets)
    tprec = (torch.sum(skel_pred * targets, dim=(1, 2, 3)) + 1e-5) / (torch.sum(skel_pred, dim=(1, 2, 3)) + 1e-5)
    tsens = (torch.sum(skel_true * probs, dim=(1, 2, 3)) + 1e-5) / (torch.sum(skel_true, dim=(1, 2, 3)) + 1e-5)
    soft_cldice = (2.0 * tprec * tsens) / (tprec + tsens + 1e-5)
    cldice_loss = (1.0 - soft_cldice).mean()
    return (self.bce_weight * bce) + (self.cldice_weight * cldice_loss)

In [ ]:
optimizer_p1 = AdamW(model_p1.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler_p1 = CosineAnnealingLR(optimizer_p1, T_max=20, eta_min=1e-6)
criterion_p1 = HybridSoftClDiceLoss(bce_weight=1.0, cldice_weight=3.0, pos_weight=1.5).to(device)

In [ ]:
optimizer_p2 = AdamW([
    {'params': model_p2.encoder.parameters(), 'lr': 5e-5, 'weight_decay': 1e-4},
    {'params': model_p2.decoder.parameters(), 'lr': 3e-4, 'weight_decay': 1e-4},
    {'params': model_p2.segmentation_head.parameters(), 'lr': 3e-4, 'weight_decay': 1e-4},
])
scheduler_p2 = CosineAnnealingLR(optimizer_p2, T_max=20, eta_min=1e-6)
criterion_p2 = HybridSoftClDiceLoss(bce_weight=1.0, cldice_weight=3.0, pos_weight=1.5).to(device)

In [ ]:
for param in model_p3.encoder.parameters():
    param.requires_grad = False

optimizer_p3 = AdamW([
    {'params': model_p3.encoder.parameters(), 'lr': 1e-5, 'weight_decay': 0.01},
    {'params': model_p3.decoder.parameters(), 'lr': 2e-4, 'weight_decay': 1e-4},
    {'params': model_p3.segmentation_head.parameters(), 'lr': 2e-4, 'weight_decay': 1e-4},
])
scheduler_p3 = CosineAnnealingLR(optimizer_p3, T_max=20, eta_min=1e-6)
criterion_p3 = HybridSoftClDiceLoss(bce_weight=1.0, cldice_weight=3.0, pos_weight=1.5).to(device)

### Evaluation function

In [ ]:
def predict_multiscale_tta(model, img_tensor, scales=(1.0,)):
  B, C, H, W = img_tensor.shape
  final_prob = torch.zeros((B, 1, H, W), device=img_tensor.device)

  for s in scales:
    if s != 1.0:
      scaled_h, scaled_w = int(H * s), int(W * s)
      scaled_img = F.interpolate(img_tensor, size=(scaled_h, scaled_w), mode='bilinear', align_corners=False)
    else:
      scaled_img = img_tensor

    scale_prob = torch.zeros_like(scaled_img[:, :1])
    for k in [0, 1, 2, 3]:
      rot = torch.rot90(scaled_img, k, dims=[-2, -1])
      scale_prob += torch.rot90(torch.sigmoid(model(rot)), -k, dims=[-2, -1])
      flip_rot = torch.flip(rot, dims=[-1])
      scale_prob += torch.rot90(torch.flip(torch.sigmoid(model(flip_rot)), dims=[-1]), -k, dims=[-2, -1])
    scale_prob /= 8.0

    if s != 1.0:
      scale_prob = F.interpolate(scale_prob, size=(H, W), mode='bilinear', align_corners=False)

    final_prob += scale_prob

  return final_prob / len(scales)

In [ ]:
def evaluate_cldice_patches(model, loader, device, low_th=0.04, high_th=0.20, min_size=20, scales=(1.0,)):
  model.eval()
  cldices, recalls, precisions = [], [], []

  with torch.inference_mode():
    for imgs, msks in loader:
      imgs = imgs.to(device, non_blocking=True)
      msks_np = (msks.squeeze(1).numpy() > 0.5).astype(np.uint8)
      probs = predict_multiscale_tta(model, imgs, scales=scales).cpu().numpy()

      for i in range(imgs.size(0)):
        m = msks_np[i]
        if m.sum() > 0:
          hyst = apply_hysteresis_threshold(probs[i, 0], low=low_th, high=high_th)
          p_clean = remove_small_objects(hyst, min_size=min_size).astype(np.uint8)

          skel_t = skeletonize(m > 0)
          skel_p = skeletonize(p_clean > 0)
          if skel_t.sum() == 0:
            continue

          sens = (skel_t * p_clean).sum() / (skel_t.sum() + 1e-7)
          dist_t = ndi.distance_transform_edt(1 - m)
          prec = (skel_p * (dist_t <= 2.5)).sum() / (skel_p.sum() + 1e-7) if skel_p.sum() > 0 else 0.0

          recalls.append(float(sens))
          precisions.append(float(prec))
          cldices.append(float((2.0 * prec * sens) / (prec + sens + 1e-7)) if (prec + sens) > 0 else 0.0)

  return float(np.mean(cldices)), float(np.mean(recalls)), float(np.mean(precisions))

### Train and Validation Step

In [ ]:
def train_step(model, dataloader, optimizer, criterion, device):
  model.train()
  total_loss = 0.0
  count = 0

  for imgs, msks in dataloader:
    imgs = imgs.to(device, non_blocking=True)
    msks = msks.to(device, non_blocking=True)

    optimizer.zero_grad()
    logits = model(imgs)
    loss = criterion(logits, msks)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    total_loss += loss.item() * imgs.size(0)
    count += imgs.size(0)

  return total_loss / count

In [ ]:
def validate_step(model, dataloader, device, low_th=0.04, high_th=0.20, min_size=20):
  model.eval()
  cldice, recall, precision = evaluate_cldice_patches(
      model, dataloader, device, low_th=low_th, high_th=high_th, min_size=min_size
  )
  return cldice, recall, precision

### Save and Load Function

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_cldice, save_path):
  checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_cldice': best_cldice
  }
  torch.save(checkpoint, save_path)
  print(f'Checkpoint saved successfully to: {save_path}')

In [ ]:
def load_checkpoint(load_path, model, optimizer=None, scheduler=None, device='cpu'):
  if not os.path.exists(load_path):
    raise FileNotFoundError(f'No checkpoint found at: {load_path}')

  checkpoint = torch.load(load_path, map_location=device)
  model.load_state_dict(checkpoint['model_state_dict'])

  if optimizer is not None and 'optimizer_state_dict' in checkpoint:
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

  if scheduler is not None and 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

  print(f'Checkpoint loaded from: {load_path} (Epoch {checkpoint.get('epoch', 'N/A')})')
  return checkpoint.get('best_cldice', 0.0)

### Training Loop

In [ ]:
epochs = 20
save_path_p1 = os.path.join(local_model_dir, 'best_resnet34_model.pth')
best_val_cldice_p1 = 0.0

print("\n--- Training Pipeline 1 (ResNet-34) ---")
print(f"{'Epoch':<10} | {'Train Loss':<12} | {'Val clDice':<12} | {'Recall':<12} | {'Precision':<12}")
print("-" * 85)

for epoch in tqdm(range(1, epochs + 1)):
  train_loss = train_step(model_p1, train_dataloader, optimizer_p1, criterion_p1, device)
  scheduler_p1.step()
  val_cldice, val_rec, val_prec = validate_step(model_p1, val_dataloader, device, low_th=0.15, high_th=0.50, min_size=20)

  tag = ""
  if val_cldice > best_val_cldice_p1:
    best_val_cldice_p1 = val_cldice
    save_checkpoint(model_p1, optimizer_p1, scheduler_p1, epoch, best_val_cldice_p1, save_path_p1)
    tag = '* [SAVED BEST]'

  print(f'[{epoch:02d}/20] | {train_loss:<12.4f} | {val_cldice:<12.4f} | {val_rec:<12.4f} | {val_prec:<12.4f} {tag}')


--- Training Pipeline 1 (ResNet-34) ---
Epoch      | Train Loss   | Val clDice   | Recall       | Precision   
-------------------------------------------------------------------------------------


  0%|          | 0/20 [00:00<?, ?it/s]

Checkpoint saved successfully to: /content/project/tnt/model/best_resnet34_model.pth
[01/20] | 3.2862       | 0.0361       | 0.5767       | 0.0282       * [SAVED BEST]
Checkpoint saved successfully to: /content/project/tnt/model/best_resnet34_model.pth
[02/20] | 2.4025       | 0.1168       | 0.5752       | 0.0731       * [SAVED BEST]
[03/20] | 1.8665       | 0.1148       | 0.6621       | 0.0659       
Checkpoint saved successfully to: /content/project/tnt/model/best_resnet34_model.pth
[04/20] | 1.5749       | 0.1549       | 0.6834       | 0.0928       * [SAVED BEST]
Checkpoint saved successfully to: /content/project/tnt/model/best_resnet34_model.pth
[05/20] | 1.3844       | 0.2315       | 0.7325       | 0.1557       * [SAVED BEST]
Checkpoint saved successfully to: /content/project/tnt/model/best_resnet34_model.pth
[06/20] | 1.2491       | 0.3386       | 0.7276       | 0.2468       * [SAVED BEST]
Checkpoint saved successfully to: /content/project/tnt/model/best_resnet34_model.pth
[07/20

In [ ]:
epochs = 20
save_path_p2 = os.path.join(local_model_dir, 'best_convnext_model.pth')
best_val_cldice_p2 = 0.0

print("\n--- Training Pipeline 2 (ConvNeXt-Nano) ---")
print(f"{'Epoch':<10} | {'Train Loss':<12} | {'Val clDice':<12} | {'Recall':<12} | {'Precision':<12}")
print("-" * 85)

for epoch in tqdm(range(1, epochs + 1)):
  train_loss = train_step(model_p2, train_dataloader, optimizer_p2, criterion_p2, device)
  scheduler_p2.step()
  val_cldice, val_rec, val_prec = validate_step(model_p2, val_dataloader, device, low_th=0.15, high_th=0.55, min_size=20)

  tag = ""
  if val_cldice > best_val_cldice_p2:
    best_val_cldice_p2 = val_cldice
    save_checkpoint(model_p2, optimizer_p2, scheduler_p2, epoch, best_val_cldice_p2, save_path_p2)
    tag = '* [SAVED BEST]'

  print(f'[{epoch:02d}/20] | {train_loss:<12.4f} | {val_cldice:<12.4f} | {val_rec:<12.4f} | {val_prec:<12.4f} {tag}')


--- Training Pipeline 2 (ConvNeXt-Nano) ---
Epoch      | Train Loss   | Val clDice   | Recall       | Precision   
-------------------------------------------------------------------------------------


  0%|          | 0/20 [00:00<?, ?it/s]

Checkpoint saved successfully to: /content/project/tnt/model/best_convnext_model.pth
[01/20] | 2.9061       | 0.1763       | 0.6293       | 0.1130       * [SAVED BEST]
[02/20] | 2.1214       | 0.1693       | 0.5802       | 0.1253       
Checkpoint saved successfully to: /content/project/tnt/model/best_convnext_model.pth
[03/20] | 1.7481       | 0.4487       | 0.5914       | 0.4054       * [SAVED BEST]
[04/20] | 1.4781       | 0.4074       | 0.6610       | 0.3294       
[05/20] | 1.2463       | 0.4122       | 0.6610       | 0.3385       
[06/20] | 1.1406       | 0.3667       | 0.6344       | 0.2946       
[07/20] | 1.0086       | 0.3889       | 0.7416       | 0.2998       
Checkpoint saved successfully to: /content/project/tnt/model/best_convnext_model.pth
[08/20] | 0.9079       | 0.4804       | 0.6854       | 0.4301       * [SAVED BEST]
[09/20] | 0.8695       | 0.4497       | 0.6979       | 0.3854       
[10/20] | 0.7699       | 0.4678       | 0.6870       | 0.4235       
[11/20] | 0.7

In [ ]:
epochs = 20
save_path_p3 = os.path.join(local_model_dir, 'best_transformer_model.pth')
best_val_cldice_p3 = 0.0

print("\n--- Training Pipeline 3 (SegFormer-B3 Vision Transformer - Optimized) ---")
print(f"{'Epoch':<10} | {'Train Loss':<12} | {'Val clDice':<12} | {'Recall':<12} | {'Precision':<12}")
print("-" * 85)

for epoch in tqdm(range(1, epochs + 1)):
  if epoch == 6:
    print("\n[Unfreezing SegFormer-B3 Encoder for fine-tuning...]")
    for param in model_p3.encoder.parameters():
      param.requires_grad = True

  train_loss = train_step(model_p3, train_dataloader, optimizer_p3, criterion_p3, device)
  scheduler_p3.step()

  val_cldice, val_rec, val_prec = validate_step(model_p3, val_dataloader, device, low_th=0.20, high_th=0.60, min_size=20)

  tag = ""
  if val_cldice > best_val_cldice_p3:
    best_val_cldice_p3 = val_cldice
    save_checkpoint(model_p3, optimizer_p3, scheduler_p3, epoch, best_val_cldice_p3, save_path_p3)
    tag = '* [SAVED BEST]'

  print(f'[{epoch:02d}/20] | {train_loss:<12.4f} | {val_cldice:<12.4f} | {val_rec:<12.4f} | {val_prec:<12.4f} {tag}')


--- Training Pipeline 3 (SegFormer-B3 Vision Transformer - Optimized) ---
Epoch      | Train Loss   | Val clDice   | Recall       | Precision   
-------------------------------------------------------------------------------------


  0%|          | 0/20 [00:00<?, ?it/s]

Checkpoint saved successfully to: /content/project/tnt/model/best_transformer_model.pth
[01/20] | 2.8350       | 0.3209       | 0.5857       | 0.2610       * [SAVED BEST]
[02/20] | 2.3067       | 0.2466       | 0.6186       | 0.1927       
[03/20] | 2.0955       | 0.2720       | 0.6634       | 0.2121       
[04/20] | 1.9496       | 0.2678       | 0.6269       | 0.2164       
[05/20] | 1.8178       | 0.2792       | 0.6319       | 0.2279       

[Unfreezing SegFormer-B3 Encoder for fine-tuning...]
Checkpoint saved successfully to: /content/project/tnt/model/best_transformer_model.pth
[06/20] | 1.6112       | 0.3420       | 0.7051       | 0.2868       * [SAVED BEST]
Checkpoint saved successfully to: /content/project/tnt/model/best_transformer_model.pth
[07/20] | 1.4437       | 0.3525       | 0.7273       | 0.3055       * [SAVED BEST]
Checkpoint saved successfully to: /content/project/tnt/model/best_transformer_model.pth
[08/20] | 1.3806       | 0.3899       | 0.7476       | 0.3431       *

### Ensemble method and Geodestic Reconstruction

In [ ]:
def optimize_geodesic_thresholds(model_boundary, model_seed, val_loader, device):
  best_th_b, best_th_s, best_score = 0.20, 0.50, 0.0
  with torch.inference_mode():
    for imgs, msks in val_loader:
      imgs = imgs.to(device, non_blocking=True)
      msks_np = (msks.squeeze(1).numpy() > 0.5).astype(np.uint8)
      prob_b = predict_multiscale_tta(model_boundary, imgs, scales=(1.0,)).cpu().numpy()
      prob_s = predict_multiscale_tta(model_seed, imgs, scales=(1.0,)).cpu().numpy()

      for tb in [0.15, 0.20, 0.25]:
        for ts in [0.40, 0.45, 0.50, 0.55]:
          scores = []
          for i in range(imgs.size(0)):
            m = msks_np[i]
            if m.sum() > 0:
              bound = (prob_b[i, 0] >= tb)
              seed = np.logical_and(remove_small_objects((prob_s[i, 0] >= ts), min_size=5), bound)
              recon = reconstruction(seed, bound).astype(np.uint8) if seed.sum() > 0 and bound.sum() > 0 else seed.astype(np.uint8)

              skel_t = skeletonize(m > 0)
              skel_p = skeletonize(recon > 0)
              if skel_t.sum() > 0:
                sens = (skel_t * recon).sum() / (skel_t.sum() + 1e-7)
                dist_t = ndi.distance_transform_edt(1 - m)
                prec = (skel_p * (dist_t <= 2.5)).sum() / (skel_p.sum() + 1e-7) if skel_p.sum() > 0 else 0.0
                sc = (2.0 * prec * sens) / (prec + sens + 1e-7) if (prec + sens) > 0 else 0.0
                scores.append(sc)
          mean_sc = np.mean(scores) if scores else 0.0
          if mean_sc > best_score:
            best_score, best_th_b, best_th_s = mean_sc, tb, ts

  return best_th_b, best_th_s

In [ ]:
print("Searching for optimal geodesic thresholds on validation set...")
opt_bound_th, opt_seed_th = optimize_geodesic_thresholds(model_p2, model_p2, val_dataloader, device)
print(f"Discovered Optimal Boundary Threshold: {opt_bound_th}")
print(f"Discovered Optimal Seed Threshold: {opt_seed_th}")

Searching for optimal geodesic thresholds on validation set...
Discovered Optimal Boundary Threshold: 0.2
Discovered Optimal Seed Threshold: 0.5


In [ ]:
def generate_ensemble_mask(prob_map_p1, prob_map_p2, low=0.25, high=0.35):
  ensemble_prob = (0.4 * prob_map_p1) + (0.6 * prob_map_p2)
  hyst_ensemble = apply_hysteresis_threshold(ensemble_prob, low=low, high=high)
  mask_ensemble = remove_small_objects(hyst_ensemble, min_size=20).astype(np.uint8)
  skel_ensemble = skeletonize(mask_ensemble > 0)
  return mask_ensemble, skel_ensemble

In [ ]:
with torch.inference_mode():
  p1_probs_list, p2_probs_list, p3_probs_list, masks_list = [], [], [], []
  for imgs, msks in test_dataloader:
    imgs = imgs.to(device, non_blocking=True)
    prob_p1 = predict_multiscale_tta(model_p1, imgs, scales=(0.75, 1.0, 1.25)).cpu().numpy()
    prob_p2 = predict_multiscale_tta(model_p2, imgs, scales=(0.75, 1.0, 1.25)).cpu().numpy()
    prob_p3 = predict_multiscale_tta(model_p3, imgs, scales=(0.75, 1.0, 1.25)).cpu().numpy()
    p1_probs_list.append(prob_p1)
    p2_probs_list.append(prob_p2)
    p3_probs_list.append(prob_p3)
    masks_list.append(msks.numpy())

ensemble_cldices, ensemble_recalls, ensemble_precisions = [], [], []
p5_cldices, p5_recalls, p5_precisions = [], [], []
p6_cldices, p6_recalls, p6_precisions = [], [], []
p7_cldices, p7_recalls, p7_precisions = [], [], []

with torch.inference_mode():
  for p1_batch, p2_batch, p3_batch, msk_batch in zip(p1_probs_list, p2_probs_list, p3_probs_list, masks_list):
    msks_np = (msk_batch.squeeze(1) > 0.5).astype(np.uint8)
    for i in range(p1_batch.shape[0]):
      m = msks_np[i]
      if m.sum() > 0:
        skel_t = skeletonize(m > 0)
        dist_t = ndi.distance_transform_edt(1 - m)

        ens_mask, _ = generate_ensemble_mask(p1_batch[i, 0], p2_batch[i, 0])
        skel_ens = skeletonize(ens_mask > 0)
        if skel_t.sum() > 0:
          sens_ens = (skel_t * (ens_mask > 0)).sum() / (skel_t.sum() + 1e-7)
          prec_ens = (skel_ens * (dist_t <= 2.5)).sum() / (skel_ens.sum() + 1e-7) if skel_ens.sum() > 0 else 0.0
          ensemble_recalls.append(float(sens_ens))
          ensemble_precisions.append(float(prec_ens))
          ensemble_cldices.append(float((2.0 * prec_ens * sens_ens) / (prec_ens + sens_ens + 1e-7)) if (prec_ens + sens_ens) > 0 else 0.0)

        prob_p2 = p2_batch[i, 0]
        bound_p5 = (prob_p2 >= opt_bound_th).astype(bool)
        seed_p5 = remove_small_objects((prob_p2 >= opt_seed_th), min_size=5).astype(bool)
        seed_p5 = np.logical_and(seed_p5, bound_p5)

        if seed_p5.sum() > 0 and bound_p5.sum() > 0:
            recon_p5 = reconstruction(seed_p5, bound_p5).astype(np.uint8)
        else:
            recon_p5 = seed_p5.astype(np.uint8)
        skel_p5 = skeletonize(recon_p5 > 0)

        if skel_t.sum() > 0:
          sens_p5 = (skel_t * recon_p5).sum() / (skel_t.sum() + 1e-7)
          prec_p5 = (skel_p5 * (dist_t <= 2.5)).sum() / (skel_p5.sum() + 1e-7) if skel_p5.sum() > 0 else 0.0
          p5_recalls.append(float(sens_p5))
          p5_precisions.append(float(prec_p5))
          p5_cldices.append(float((2.0 * prec_p5 * sens_p5) / (prec_p5 + sens_p5 + 1e-7)) if (prec_p5 + sens_p5) > 0 else 0.0)

        prob_p3 = p3_batch[i, 0]
        bound_p6 = (prob_p3 >= opt_bound_th).astype(bool)
        seed_p6 = seed_p5.copy()
        seed_p6 = np.logical_and(seed_p6, bound_p6)

        if seed_p6.sum() > 0 and bound_p6.sum() > 0:
            recon_p6 = reconstruction(seed_p6, bound_p6).astype(np.uint8)
        else:
            recon_p6 = seed_p6.astype(np.uint8)
        skel_p6 = skeletonize(recon_p6 > 0)

        if skel_t.sum() > 0:
          sens_p6 = (skel_t * recon_p6).sum() / (skel_t.sum() + 1e-7)
          prec_p6 = (skel_p6 * (dist_t <= 2.5)).sum() / (skel_p6.sum() + 1e-7) if skel_p6.sum() > 0 else 0.0
          p6_recalls.append(float(sens_p6))
          p6_precisions.append(float(prec_p6))
          p6_cldices.append(float((2.0 * prec_p6 * sens_p6) / (prec_p6 + sens_p6 + 1e-7)) if (prec_p6 + sens_p6) > 0 else 0.0)

        prob_ens_geo = (0.5 * prob_p2) + (0.5 * prob_p3)
        bound_p7 = (prob_ens_geo >= opt_bound_th).astype(bool)
        seed_p7 = np.logical_and(remove_small_objects((prob_ens_geo >= opt_seed_th), min_size=5), bound_p7)

        if seed_p7.sum() > 0 and bound_p7.sum() > 0:
            recon_p7 = reconstruction(seed_p7, bound_p7).astype(np.uint8)
        else:
            recon_p7 = seed_p7.astype(np.uint8)
        skel_p7 = skeletonize(recon_p7 > 0)

        if skel_t.sum() > 0:
          sens_p7 = (skel_t * recon_p7).sum() / (skel_t.sum() + 1e-7)
          prec_p7 = (skel_p7 * (dist_t <= 2.5)).sum() / (skel_p7.sum() + 1e-7) if skel_p7.sum() > 0 else 0.0
          p7_recalls.append(float(sens_p7))
          p7_precisions.append(float(prec_p7))
          p7_cldices.append(float((2.0 * prec_p7 * sens_p7) / (prec_p7 + sens_p7 + 1e-7)) if (prec_p7 + sens_p7) > 0 else 0.0)

test_cldice_p4 = float(np.mean(ensemble_cldices))
test_rec_p4 = float(np.mean(ensemble_recalls))
test_prec_p4 = float(np.mean(ensemble_precisions))

test_cldice_p5 = float(np.mean(p5_cldices))
test_rec_p5 = float(np.mean(p5_recalls))
test_prec_p5 = float(np.mean(p5_precisions))

test_cldice_p6 = float(np.mean(p6_cldices))
test_rec_p6 = float(np.mean(p6_recalls))
test_prec_p6 = float(np.mean(p6_precisions))

test_cldice_p7 = float(np.mean(p7_cldices))
test_rec_p7 = float(np.mean(p7_recalls))
test_prec_p7 = float(np.mean(p7_precisions))

### Evaluation Result

In [ ]:
load_checkpoint(save_path_p1, model_p1, device=device)
test_cldice_p1, test_rec_p1, test_prec_p1 = evaluate_cldice_patches(model_p1, test_dataloader, device, low_th=0.15, high_th=0.50, min_size=20, scales=(0.75, 1.0, 1.25))

load_checkpoint(save_path_p2, model_p2, device=device)
test_cldice_p2, test_rec_p2, test_prec_p2 = evaluate_cldice_patches(model_p2, test_dataloader, device, low_th=0.15, high_th=0.55, min_size=20, scales=(0.75, 1.0, 1.25))

load_checkpoint(save_path_p3, model_p3, device=device)
test_cldice_p3, test_rec_p3, test_prec_p3 = evaluate_cldice_patches(model_p3, test_dataloader, device, low_th=0.20, high_th=0.60, min_size=20, scales=(0.75, 1.0, 1.25))

print('\n' + '='*105)
print('FINAL COMPREHENSIVE COMPARATIVE EVALUATION ON TEST PATCHES (ALL 7 METHODS)')
print('='*105)
print(f"{'Method / Architecture':<50} | {'clDice Score':<14} | {'Recall':<12} | {'Precision':<12}")
print("-" * 105)
print(f"{'1. ResNet-34 U-Net (Pipeline 1)':<50} | {test_cldice_p1:<14.4f} | {test_rec_p1:<12.4f} | {test_prec_p1:<12.4f}")
print(f"{'2. ConvNeXt-Nano U-Net (Pipeline 2)':<50} | {test_cldice_p2:<14.4f} | {test_rec_p2:<12.4f} | {test_prec_p2:<12.4f}")
print(f"{'3. SegFormer-B3 Transformer (Pipeline 3)':<50} | {test_cldice_p3:<14.4f} | {test_rec_p3:<12.4f} | {test_prec_p3:<12.4f}")
print(f"{'4. Ensemble Model (ResNet + ConvNeXt)':<50} | {test_cldice_p4:<14.4f} | {test_rec_p4:<12.4f} | {test_prec_p4:<12.4f}")
print(f"{'5. Geodesic Reconstruction ConvNeXt':<50} | {test_cldice_p5:<14.4f} | {test_rec_p5:<12.4f} | {test_prec_p5:<12.4f}")
print(f"{'6. SegFormer-Guided Geodesic Reconstruction':<50} | {test_cldice_p6:<14.4f} | {test_rec_p6:<12.4f} | {test_prec_p6:<12.4f}")
print(f"{'7. Probability-Ensemble Guided Geodesic Reconstruction':<50} | {test_cldice_p7:<14.4f} | {test_rec_p7:<12.4f} | {test_prec_p7:<12.4f}")
print('='*105)

Checkpoint loaded from: /content/project/tnt/model/best_resnet34_model.pth (Epoch 14)
Checkpoint loaded from: /content/project/tnt/model/best_convnext_model.pth (Epoch 8)
Checkpoint loaded from: /content/project/tnt/model/best_transformer_model.pth (Epoch 19)

FINAL COMPREHENSIVE COMPARATIVE EVALUATION ON TEST PATCHES (ALL 7 METHODS)
Method / Architecture                              | clDice Score   | Recall       | Precision   
---------------------------------------------------------------------------------------------------------
1. ResNet-34 U-Net (Pipeline 1)                    | 0.4125         | 0.6003       | 0.3433      
2. ConvNeXt-Nano U-Net (Pipeline 2)                | 0.4521         | 0.5616       | 0.4409      
3. SegFormer-B3 Transformer (Pipeline 3)           | 0.4091         | 0.6616       | 0.3273      
4. Ensemble Model (ResNet + ConvNeXt)              | 0.4904         | 0.5788       | 0.5038      
5. Geodesic Reconstruction ConvNeXt                | 0.4609         

In [ ]:
def evaluate_and_visualize_pipeline(
    model_or_func,
    pipeline_name,
    dataset,
    test_cldice,
    test_rec,
    test_prec,
    is_ensemble=False,
    is_prob_ensemble_geo=False,
    model_p1=None,
    model_p2=None,
    model_p3=None,
    fixed_sample_idx=0,
):
  sample_img_tensor, sample_mask_tensor = dataset[fixed_sample_idx]

  with torch.inference_mode():
    if is_ensemble:
      prob_p1 = predict_multiscale_tta(model_p1, sample_img_tensor.unsqueeze(0).to(device), scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
      prob_p2 = predict_multiscale_tta(model_p2, sample_img_tensor.unsqueeze(0).to(device), scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
      sample_prob = (0.4 * prob_p1) + (0.6 * prob_p2)
    elif is_prob_ensemble_geo:
      prob_p2 = predict_multiscale_tta(model_p2, sample_img_tensor.unsqueeze(0).to(device), scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
      prob_p3 = predict_multiscale_tta(model_p3, sample_img_tensor.unsqueeze(0).to(device), scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
      sample_prob = (0.5 * prob_p2) + (0.5 * prob_p3)
    else:
      sample_prob = predict_multiscale_tta(model_or_func, sample_img_tensor.unsqueeze(0).to(device), scales=(0.75, 1.0, 1.25))
      sample_prob = sample_prob.squeeze().cpu().numpy()

  sample_img_np = sample_img_tensor[0].cpu().numpy()
  sample_img_scaled = ((sample_img_np - sample_img_np.min()) / (np.ptp(sample_img_np) + 1e-7) * 255).astype(np.uint8)

  if sample_mask_tensor.ndim == 3:
    sample_mask_np = sample_mask_tensor.squeeze(0).cpu().numpy()
  else:
    sample_mask_np = sample_mask_tensor.cpu().numpy()

  if is_ensemble:
    sample_clean_pred, _ = generate_ensemble_mask(prob_p1, prob_p2)
  elif ('Geodesic Reconstruction' in pipeline_name or 'Probability-Ensemble Guided' in pipeline_name):
    bound_mask = (sample_prob >= opt_bound_th).astype(bool)
    seed_mask = remove_small_objects((sample_prob >= opt_seed_th), min_size=5).astype(bool)
    seed_mask = np.logical_and(seed_mask, bound_mask)

    if seed_mask.sum() > 0 and bound_mask.sum() > 0:
      sample_clean_pred = reconstruction(seed_mask, bound_mask).astype(np.uint8)
    else:
      sample_clean_pred = seed_mask.astype(np.uint8)
  elif 'SegFormer-Guided' in pipeline_name:
    prob_p2_single = predict_multiscale_tta(model_p2, sample_img_tensor.unsqueeze(0).to(device), scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
    bound_mask = (sample_prob >= opt_bound_th).astype(bool)
    seed_mask = remove_small_objects((prob_p2_single >= opt_seed_th), min_size=5).astype(bool)
    seed_mask = np.logical_and(seed_mask, bound_mask)

    if seed_mask.sum() > 0 and bound_mask.sum() > 0:
      sample_clean_pred = reconstruction(seed_mask, bound_mask).astype(np.uint8)
    else:
      sample_clean_pred = seed_mask.astype(np.uint8)
  else:
    if 'Pipeline 1' in pipeline_name:
      low_th, high_th = 0.15, 0.50
    elif 'Pipeline 2' in pipeline_name:
      low_th, high_th = 0.15, 0.55
    elif 'Pipeline 3' in pipeline_name:
      low_th, high_th = 0.20, 0.60
    else:
      low_th, high_th = 0.15, 0.55

    sample_hyst = apply_hysteresis_threshold(sample_prob, low=low_th, high=high_th)
    sample_clean_pred = remove_small_objects(sample_hyst, min_size=20).astype(np.uint8)

  print(
      f'\n{pipeline_name} Test Patch #{fixed_sample_idx} -> GT Pixels:'
      f' {np.sum(sample_mask_np > 0)} | Pred Pixels:'
      f' {np.sum(sample_clean_pred > 0)}'
  )

  canvas_rgb = cv2.cvtColor(sample_img_scaled, cv2.COLOR_GRAY2RGB)
  overlay = canvas_rgb.copy()
  gt_bool = sample_mask_np > 0.5
  pred_bool = sample_clean_pred > 0

  overlay[gt_bool] = [0, 200, 0]
  overlay[pred_bool] = [200, 0, 200]
  blended_overlay = cv2.addWeighted(canvas_rgb, 0.5, overlay, 0.5, 0)

  fig, axes = plt.subplots(1, 4, figsize=(20, 6), dpi=300)
  axes[0].imshow(sample_img_scaled, cmap='gray')
  axes[0].set_title(f'1. Test Patch #{fixed_sample_idx}', fontsize=10)
  axes[0].axis('off')

  axes[1].imshow(sample_mask_np, cmap='gray', vmin=0, vmax=1)
  axes[1].set_title('2. Ground Truth Mask', fontsize=10)
  axes[1].axis('off')

  im = axes[2].imshow(sample_prob, cmap='inferno', vmin=0, vmax=1)
  axes[2].set_title('3. TTA Probability Map', fontsize=10)
  axes[2].axis('off')
  fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

  axes[3].imshow(blended_overlay)
  axes[3].set_title(f'4. Overlay (GT: Green | Pred: Magenta)\nTest clDice: {test_cldice:.4f}', fontsize=10)
  axes[3].axis('off')

  plt.tight_layout()
  plot_filename = ('test_patch_evaluation_plot_'+ pipeline_name.lower().replace(' ', '_').replace('.', '')+ '.png')
  plot_save_path = os.path.join(local_plot_dir, plot_filename)
  plt.savefig(plot_save_path, bbox_inches='tight')
  plt.show()

  print('=' * 85)
  print(f'{pipeline_name} Quantitative Report:')
  print('=' * 85)
  print(f'Centerline Dice (clDice @ 2.5px tolerance): {test_cldice * 100:.2f}%')
  print(f'Skeleton Recall (Sensitivity):               {test_rec * 100:.2f}%')
  print(f'Skeleton Precision:                          {test_prec * 100:.2f}%')
  print(f'Total Positive Tubule Pixels (Pred):         {np.sum(sample_clean_pred > 0):,}')
  print(f'Total Positive Tubule Pixels (GT):           {np.sum(sample_mask_np > 0):,}')
  print('=' * 85)

In [ ]:
valid_indices = [idx for idx, (_, m_tensor) in enumerate(test_dataset) if (m_tensor > 0.5).sum() > 50]
target_sample_idx = random.choice(valid_indices) if valid_indices else 0

In [2]:
# Hide due to data privacy

# evaluate_and_visualize_pipeline(model_p1, '1. ResNet-34 U-Net (Pipeline 1)', test_dataset, test_cldice_p1, test_rec_p1, test_prec_p1, fixed_sample_idx=target_sample_idx)
# evaluate_and_visualize_pipeline(model_p2, '2. ConvNeXt-Nano U-Net (Pipeline 2)', test_dataset, test_cldice_p2, test_rec_p2, test_prec_p2, fixed_sample_idx=target_sample_idx)
# evaluate_and_visualize_pipeline(model_p3, '3. SegFormer-B3 Transformer (Pipeline 3)', test_dataset, test_cldice_p3, test_rec_p3, test_prec_p3, fixed_sample_idx=target_sample_idx)
# evaluate_and_visualize_pipeline(None, '4. Ensemble Model (ResNet + ConvNeXt)', test_dataset, test_cldice_p4, test_rec_p4, test_prec_p4, is_ensemble=True, model_p1=model_p1, model_p2=model_p2, fixed_sample_idx=target_sample_idx)
# evaluate_and_visualize_pipeline(model_p2, '5. Geodesic Reconstruction ConvNeXt', test_dataset, test_cldice_p5, test_rec_p5, test_prec_p5, fixed_sample_idx=target_sample_idx)
# evaluate_and_visualize_pipeline(model_p3, '6. SegFormer-Guided Geodesic Reconstruction', test_dataset, test_cldice_p6, test_rec_p6, test_prec_p6, model_p2=model_p2, fixed_sample_idx=target_sample_idx)
# evaluate_and_visualize_pipeline(None, '7. Probability-Ensemble Guided Geodesic Reconstruction', test_dataset, test_cldice_p7, test_rec_p7, test_prec_p7, is_prob_ensemble_geo=True, model_p2=model_p2, model_p3=model_p3, fixed_sample_idx=target_sample_idx)

### TNT count recovery

In [ ]:
def count_and_evaluate_individual_tnts(pred_mask, gt_mask):
  gt_skel = skeletonize(gt_mask > 0)
  gt_labeled, gt_num = ndi.label(gt_skel)

  pred_clean = remove_small_objects(pred_mask > 0, min_size=20)
  pred_labeled, pred_num = ndi.label(pred_clean)

  recovered_count = 0
  for i in range(1, gt_num + 1):
    if np.logical_and(pred_clean, gt_labeled == i).sum() > 0:
      recovered_count += 1
  instance_recall = recovered_count / (gt_num + 1e-7)

  matched_pred_count = 0
  for j in range(1, pred_num + 1):
    if np.logical_and(gt_mask > 0, pred_labeled == j).sum() > 0:
      matched_pred_count += 1
  instance_precision = matched_pred_count / (pred_num + 1e-7)

  return gt_num, pred_num, recovered_count, instance_recall, instance_precision

In [ ]:
def visualize_whole_image_predictions(full_img_raw, full_gt_mask, pipelines_dict, output_dir, dilate_kernel_size=3):
  os.makedirs(output_dir, exist_ok=True)
  img_normalized = ((full_img_raw - full_img_raw.min()) / (np.ptp(full_img_raw) + 1e-7) * 255).astype(np.uint8)
  base_rgb = cv2.cvtColor(img_normalized, cv2.COLOR_GRAY2RGB)

  kernel_small = np.ones((dilate_kernel_size, dilate_kernel_size), np.uint8)
  kernel_large = np.ones((dilate_kernel_size + 2, dilate_kernel_size + 2), np.uint8)

  gt_base = (full_gt_mask > 0).astype(np.uint8)
  gt_dilated = cv2.dilate(gt_base, kernel_small, iterations=1) > 0

  gt_overlay = base_rgb.copy()
  gt_reference_bold = cv2.dilate(gt_base, kernel_large, iterations=2) > 0
  for c in range(3):
    gt_overlay[gt_reference_bold, c] = np.array([255, 255, 0], dtype=np.uint8)[c]
  blended_gt = cv2.addWeighted(base_rgb, 0.3, gt_overlay, 0.7, 0)

  plt.figure(figsize=(14, 14), dpi=300)
  plt.imshow(blended_gt)
  plt.title("Whole-Image Ground Truth Overlay (Reference)", fontsize=14)
  plt.axis('off')
  gt_plot_path = os.path.join(output_dir, "whole_image_overlay_ground_truth.png")
  plt.savefig(gt_plot_path, bbox_inches='tight', pad_inches=0.1)
  plt.show()
  plt.close()

  for name, pred_mask in pipelines_dict:
    pred_base = (pred_mask > 0).astype(np.uint8)
    pred_dilated = cv2.dilate(pred_base, kernel_small, iterations=1) > 0

    overlay = base_rgb.copy()
    missed_gt_raw = np.logical_and(gt_dilated, np.logical_not(pred_dilated))
    false_pos_raw = np.logical_and(pred_dilated, np.logical_not(gt_dilated))
    correct_match_raw = np.logical_and(gt_dilated, pred_dilated)

    missed_gt = cv2.dilate(missed_gt_raw.astype(np.uint8), kernel_large, iterations=1) > 0
    correct_match = cv2.dilate(correct_match_raw.astype(np.uint8), kernel_large, iterations=1) > 0
    false_pos = cv2.dilate(false_pos_raw.astype(np.uint8), kernel_small, iterations=1) > 0

    COLOR_MISSED_GT = np.array([0, 255, 255], dtype=np.uint8)
    COLOR_FALSE_POSITIVE = np.array([255, 255, 0], dtype=np.uint8)
    COLOR_CORRECT_MATCH = np.array([255, 0, 255], dtype=np.uint8)

    for c in range(3):
      overlay[false_pos, c] = COLOR_FALSE_POSITIVE[c]
      overlay[missed_gt, c] = COLOR_MISSED_GT[c]
      overlay[correct_match, c] = COLOR_CORRECT_MATCH[c]

    blended = cv2.addWeighted(base_rgb, 0.4, overlay, 0.6, 0)
    plt.figure(figsize=(14, 14), dpi=300)
    plt.imshow(blended)
    plt.title(f"Error Analysis Overlay: {name}\n(Cyan = Missed GT | Yellow = False Positive | Magenta = Correct Match)", fontsize=13)
    plt.axis('off')

    safe_name = name.lower().replace(" ", "_").replace(".", "").replace("-", "_")
    plot_path = os.path.join(output_dir, f"whole_image_overlay_{safe_name}.png")
    plt.savefig(plot_path, bbox_inches='tight', pad_inches=0.1)
    plt.show()
    plt.close()

In [3]:
# Hide due to data privacy

# test_img_path, test_annot_path = dataset_splits['test']
# full_img_raw = cv2.imread(test_img_path, cv2.IMREAD_GRAYSCALE)
# full_annot_bgr = cv2.imread(test_annot_path, cv2.IMREAD_COLOR)

# lower_hsv, upper_hsv = calibrate_marker_hsv(dataset_splits['train'][1])
# full_gt_mask = binarize_annotation(full_annot_bgr, lower_hsv, upper_hsv, min_area_px=15)
# full_gt_mask = (full_gt_mask > 127).astype(np.uint8)

# H, W = full_img_raw.shape
# patch_size, stride = 512, 512

# full_prob_p1 = np.zeros((H, W), dtype=np.float32)
# full_prob_p2 = np.zeros((H, W), dtype=np.float32)
# full_prob_p3 = np.zeros((H, W), dtype=np.float32)
# count_map = np.zeros((H, W), dtype=np.float32)

# val_test_transform_raw = A.Compose([
#   A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
#   A.Normalize(mean=(0.0,), std=(1.0,)),
#   ToTensorV2()
# ])

# model_p1.eval()
# model_p2.eval()
# model_p3.eval()

# with torch.inference_mode():
#   for y in range(0, H - patch_size + 1, stride):
#     for x in range(0, W - patch_size + 1, stride):
#       patch_gray = full_img_raw[y:y+patch_size, x:x+patch_size]
#       augmented = val_test_transform_raw(image=np.expand_dims(patch_gray, axis=-1))
#       patch_tensor = augmented['image'].unsqueeze(0).to(device).repeat(1, 3, 1, 1)

#       full_prob_p1[y:y+patch_size, x:x+patch_size] += predict_multiscale_tta(model_p1, patch_tensor, scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
#       full_prob_p2[y:y+patch_size, x:x+patch_size] += predict_multiscale_tta(model_p2, patch_tensor, scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
#       full_prob_p3[y:y+patch_size, x:x+patch_size] += predict_multiscale_tta(model_p3, patch_tensor, scales=(0.75, 1.0, 1.25)).squeeze().cpu().numpy()
#       count_map[y:y+patch_size, x:x+patch_size] += 1.0

# count_map = np.maximum(count_map, 1.0)
# full_prob_p1 /= count_map
# full_prob_p2 /= count_map
# full_prob_p3 /= count_map

# pred_1 = remove_small_objects(apply_hysteresis_threshold(full_prob_p1, low=0.20, high=0.55), min_size=40).astype(np.uint8)
# pred_2 = remove_small_objects(apply_hysteresis_threshold(full_prob_p2, low=0.20, high=0.60), min_size=40).astype(np.uint8)
# pred_3 = remove_small_objects(apply_hysteresis_threshold(full_prob_p3, low=0.25, high=0.65), min_size=40).astype(np.uint8)

# ens_prob_p4 = (0.4 * full_prob_p1) + (0.6 * full_prob_p2)
# pred_4 = remove_small_objects(apply_hysteresis_threshold(ens_prob_p4, low=0.20, high=0.60), min_size=40).astype(np.uint8)

# bound_p5 = (full_prob_p2 >= opt_bound_th).astype(bool)
# seed_p5 = np.logical_and(remove_small_objects((full_prob_p2 >= opt_seed_th), min_size=10), bound_p5)
# pred_5 = reconstruction(seed_p5, bound_p5).astype(np.uint8) if seed_p5.sum() > 0 and bound_p5.sum() > 0 else seed_p5.astype(np.uint8)

# bound_p6 = (full_prob_p3 >= opt_bound_th).astype(bool)
# seed_p6 = np.logical_and(seed_p5, bound_p6)
# pred_6 = reconstruction(seed_p6, bound_p6).astype(np.uint8) if seed_p6.sum() > 0 and bound_p6.sum() > 0 else seed_p6.astype(np.uint8)

# prob_ens_geo = (0.5 * full_prob_p2) + (0.5 * full_prob_p3)
# bound_p7 = (prob_ens_geo >= opt_bound_th).astype(bool)
# seed_p7 = np.logical_and(remove_small_objects((prob_ens_geo >= opt_seed_th), min_size=10), bound_p7)
# pred_7 = reconstruction(seed_p7, bound_p7).astype(np.uint8) if seed_p7.sum() > 0 and bound_p7.sum() > 0 else seed_p7.astype(np.uint8)

# pipelines_dict = [
#   ("1. ResNet-34 U-Net", pred_1),
#   ("2. ConvNeXt-Nano U-Net", pred_2),
#   ("3. SegFormer-B3 Transformer", pred_3),
#   ("4. Ensemble Model", pred_4),
#   ("5. Geodesic Reconstruction ConvNeXt", pred_5),
#   ("6. SegFormer-Guided Geodesic", pred_6),
#   ("7. Probability-Ensemble Geodesic", pred_7)
# ]

# print('\n' + '='*105)
# print(f'WHOLE-IMAGE INSTANCE-LEVEL EVALUATION REPORT (Test Image: m03.png)')
# print('='*105)
# print(f"{'Pipeline Name':<36} | {'GT Count':<8} | {'Pred Count':<10} | {'Recovered':<10} | {'Recall':<10} | {'Precision'}")
# print("-" * 105)
# for name, p_mask in pipelines_dict:
#   gt_num, pred_num, recovered, inst_recall, inst_precision = count_and_evaluate_individual_tnts(p_mask, full_gt_mask)
#   print(f"{name:<36} | {gt_num:<8} | {pred_num:<10} | {recovered:<10} | {inst_recall * 100:.2f}%    | {inst_precision * 100:.2f}%")
# print('='*105)

# visualize_whole_image_predictions(full_img_raw, full_gt_mask, pipelines_dict, local_plot_dir, dilate_kernel_size=3)

In [4]:
# Result of above code
# =========================================================================================================
# WHOLE-IMAGE INSTANCE-LEVEL EVALUATION REPORT (Test Image: m03.png)
# =========================================================================================================
# Pipeline Name                        | GT Count | Pred Count | Recovered  | Recall     | Precision
# ---------------------------------------------------------------------------------------------------------
# 1. ResNet-34 U-Net                   | 1317     | 36         | 997        | 75.70%    | 52.78%
# 2. ConvNeXt-Nano U-Net               | 1317     | 38         | 894        | 67.88%    | 52.63%
# 3. SegFormer-B3 Transformer          | 1317     | 55         | 1044       | 79.27%    | 34.55%
# 4. Ensemble Model                    | 1317     | 34         | 945        | 71.75%    | 55.88%
# 5. Geodesic Reconstruction ConvNeXt  | 1317     | 40         | 894        | 67.88%    | 50.00%
# 6. SegFormer-Guided Geodesic         | 1317     | 36         | 1069       | 81.17%    | 52.78%
# 7. Probability-Ensemble Geodesic     | 1317     | 48         | 1044       | 79.27%    | 41.67%
# =========================================================================================================

### Save model and plot

In [ ]:
!cp -r /content/project/tnt/model/* /content/drive/MyDrive/project/tnt/model/
!cp -r /content/project/tnt/plot/* /content/drive/MyDrive/project/tnt/plot/